In [0]:
%pip install pymysql

In [0]:
from pyspark.sql.functions import *
from framework.reader import *
from framework.writer import *
from framework.audit import *
from framework.validator import *
from framework.metrics import *
from framework.mysql_connector import *
from framework.business_rule_validator import *

In [0]:
VALIDATORS = {
    "erp_validator": erp_validator,
    "crm_validator": crm_validator,
    "website_validator": website_validator,
    "inventory_validator": inventory_validator,
    "payment_validator": payment_validator
}

In [0]:
pipelines=fetch_eligible_pipelines()
if not pipelines:
    raise Exception("No pipelines")
else:
    for pipeline in pipelines:
      batch_id=None
      total_records=0
      good_records=0
      bad_records=0
      try:
        print(f"Ingestion of pipeline:{pipeline['pipeline_name']}")
        batch_id=insert_execution(pipeline["pipeline_name"])
        raw_df=read_json(spark, pipeline["raw_path"])
        clean_df,corrupt_df,corrupt_count=separate_corrupt_records(raw_df)
        validator=VALIDATORS[pipeline["validation_fun"]]
        valid_condition=validator()
        good_df,bad_df=split_records(clean_df,valid_condition)
        bronze_df=add_audit_columns_clean(good_df,pipeline["source_system"],batch_id)
        total_records,good_records,bad_records=calculate_metrics(raw_df,good_df,bad_df)
        duplicate_df=raw_df.groupBy(col(pipeline["business_key"])).count().where(col("count")>1)
        assert total_records==good_records+bad_records
        write_delta(bronze_df,pipeline["bronze_path"],"overwrite")
        bronze_bad_df=combine_bad_records(bad_df,corrupt_df,corrupt_count,batch_id,pipeline["pipeline_name"],pipeline["source_system"],pipeline["table_name"])
        write_delta(bronze_bad_df,pipeline["bad_record_path"],"append")
        update_completed(batch_id,total_records,good_records,bad_records)
        print(f"Ingestion of pipeline:{pipeline['pipeline_name']} done")
      except Exception as e:
        if batch_id is not None:
          update_failed(batch_id,total_records,good_records,bad_records,str(e)[:1000])
        else:
          print(f"Ingestion of pipeline:{pipeline['pipeline_name']} failed")